# Val Comparison: AMO (finetune_val_preds) vs dino_3_final_v0_with_val

- **A** — `amo/finetune_val_preds.csv` (pre-computed val predictions)
- **B** — `checkpoints/selected/dino_3_final_v0_with_val.pt` (inferred here on val split)

Both have ground truth → actual competition scores can be computed.

In [8]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import torch
from torch.utils.data import DataLoader

from src.dino.dino_full import build_dino_full_model, _build_image_transform
from src.dino.utils import load_config, ImageDataset
from src.config import DEVICE, NUM_WORKERS
from src.metrics import metric_fn

LABEL_A = 'AMO (finetune_val_preds)'
LABEL_B = 'dino_3_final_v0'
CKPT_B  = ROOT / 'checkpoints/selected/dino_3_final_v0_with_val.pt'
CSV_A   = ROOT / 'amo/finetune_val_preds.csv'

print(f'Device: {DEVICE}')

Device: mps


## 1. Load AMO val predictions

In [9]:
df_a = pd.read_csv(CSV_A)
# rename columns to a canonical form
df_a = df_a.rename(columns={'gt': 'FaceOcclusion', 'pred': 'pred_a'})
print(f'AMO val: {len(df_a)} rows  |  columns: {list(df_a.columns)}')
df_a.head(3)

AMO val: 20000 rows  |  columns: ['filename', 'FaceOcclusion', 'pred_a', 'gender', 'iw']


,filename,FaceOcclusion,pred_a,gender,iw
0,database3/database3/m.01wb_wm/67-FaceId-0_alig...,0.223541,0.185132,1.0,2.974629
1,database3/database3/m.01r8fq8/35-FaceId-0_alig...,0.263153,0.264450,0.0,3.593767
2,database3/database3/m.023c9m/32-FaceId-0_align...,0.273725,0.275304,1.0,3.593767


## 2. Run inference with dino_3_final_v0_with_val.pt

In [10]:
cfg_b = load_config('dino_3_final_v0.yaml')
cfg_b["head_checkpoint"] = "../checkpoints/selected/dino_cnn_2026-06-09_20-48-20_best_exp_pad_v3.pt"
model_b = build_dino_full_model(cfg_b)
model_b.load_state_dict(torch.load(CKPT_B, map_location='cpu'))
model_b = model_b.to(DEVICE)
model_b.eval()
print(f'Loaded checkpoint: {CKPT_B.name}')

transform_b = _build_image_transform(cfg_b['model_name'])
val_ds_b    = ImageDataset('val', cfg_b, transform_b, augment=False)
val_loader_b = DataLoader(
    val_ds_b, batch_size=128, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')
)
print(f'Val dataset: {len(val_ds_b)} samples')

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 10804.91it/s]


Head: PatchCNN loaded from ../checkpoints/selected/dino_cnn_2026-06-09_20-48-20_best_exp_pad_v3.pt
Loaded checkpoint: dino_3_final_v0_with_val.pt
Val dataset: 20000 samples


In [11]:
preds_b, gts_b, genders_b, iws_b, filenames_b = [], [], [], [], []

with torch.no_grad():
    offset = 0
    for img, y, gender, iw, pi, gw in val_loader_b:
        pred = model_b(img.to(DEVICE)).squeeze(-1).cpu()
        n = len(pred)
        preds_b.append(pred)
        gts_b.append(y)
        genders_b.append(gender)
        iws_b.append(iw)
        filenames_b.extend(val_ds_b.filenames[offset : offset + n])
        offset += n

df_b = pd.DataFrame({
    'filename':      filenames_b,
    'FaceOcclusion': torch.cat(gts_b).numpy(),
    'pred_b':        torch.cat(preds_b).numpy(),
    'gender':        torch.cat(genders_b).numpy(),
    'iw':            torch.cat(iws_b).numpy(),
})
print(f'Inference done: {len(df_b)} predictions')
df_b.head(3)

Inference done: 20000 predictions


,filename,FaceOcclusion,pred_b,gender,iw
0,database3/database3/m.01jxq6y/10-FaceId-0_alig...,0.003048,0.029699,1.0,0.314548
1,database3/database3/m.024jwt/34-FaceId-0_align...,0.026805,0.030451,1.0,0.314548
2,database1/img00022639.webp,0.260328,0.247451,0.0,3.593772


## 3. Merge on filename

In [12]:
df = df_a[['filename', 'FaceOcclusion', 'pred_a', 'gender', 'iw']].merge(
    df_b[['filename', 'pred_b']],
    on='filename', how='inner'
)

print(f'Merged: {len(df)} rows  (A only: {len(df_a)-len(df)}, B only: {len(df_b)-len(df)})')

gt     = df['FaceOcclusion'].values.astype(float)
pred_a = df['pred_a'].values.astype(float)
pred_b = df['pred_b'].values.astype(float)
gender = df['gender'].values.astype(float)
iw     = df['iw'].values.astype(float)
diff   = pred_a - pred_b

Merged: 20000 rows  (A only: 0, B only: 0)


## 4. Coherence checks

In [13]:
issues = []

print(f'Row count: A={len(df_a)}, B={len(df_b)}, merged={len(df)}')

for name, arr in [(LABEL_A, pred_a), (LABEL_B, pred_b)]:
    n_nan = np.isnan(arr).sum()
    n_oob = ((arr < 0) | (arr > 1)).sum()
    if n_nan == 0 and n_oob == 0:
        print(f'✓  [{name}] no NaN, all in [0, 1]')
    else:
        msg = f'✗  [{name}] {n_nan} NaN, {n_oob} out-of-range'
        issues.append(msg); print(msg)

print()
if not issues:
    print('All checks passed.')
else:
    print(f'{len(issues)} issue(s) — see above.')

Row count: A=20000, B=20000, merged=20000
✓  [AMO (finetune_val_preds)] no NaN, all in [0, 1]
✓  [dino_3_final_v0] no NaN, all in [0, 1]

All checks passed.


## 5. Competition scores (actual)

Val has ground truth → compute exact competition metric for both models.

In [14]:
gt_t  = torch.tensor(gt,     dtype=torch.float32)
g_t   = torch.tensor(gender, dtype=torch.float32)
pa_t  = torch.tensor(pred_a, dtype=torch.float32)
pb_t  = torch.tensor(pred_b, dtype=torch.float32)

score_a = metric_fn(pa_t, gt_t, g_t).item()
score_b = metric_fn(pb_t, gt_t, g_t).item()
delta   = score_a - score_b

# Per-gender breakdown
from src.metrics import error_fn
err_a_f = error_fn(pa_t[g_t == 0], gt_t[g_t == 0]).item()
err_a_m = error_fn(pa_t[g_t == 1], gt_t[g_t == 1]).item()
err_b_f = error_fn(pb_t[g_t == 0], gt_t[g_t == 0]).item()
err_b_m = error_fn(pb_t[g_t == 1], gt_t[g_t == 1]).item()

score_df = pd.DataFrame([
    {'Model': LABEL_A, 'Score': f'{score_a:.6f}', 'Err_F': f'{err_a_f:.6f}', 'Err_M': f'{err_a_m:.6f}', '|Err_F - Err_M|': f'{abs(err_a_f-err_a_m):.6f}'},
    {'Model': LABEL_B, 'Score': f'{score_b:.6f}', 'Err_F': f'{err_b_f:.6f}', 'Err_M': f'{err_b_m:.6f}', '|Err_F - Err_M|': f'{abs(err_b_f-err_b_m):.6f}'},
]).set_index('Model')

print(f'Δ score (A − B) = {delta:+.6f}  ({"A is better" if delta < 0 else "B is better"})')
display(score_df)

TypeError: metric_fn() takes 2 positional arguments but 3 were given

## 6. Summary statistics

In [ ]:
def describe(arr, label):
    return {
        'Model':  label,
        'Mean':   f'{arr.mean():.4f}',
        'Std':    f'{arr.std():.4f}',
        'Min':    f'{arr.min():.4f}',
        'p5':     f'{np.percentile(arr, 5):.4f}',
        'p25':    f'{np.percentile(arr, 25):.4f}',
        'Median': f'{np.median(arr):.4f}',
        'p75':    f'{np.percentile(arr, 75):.4f}',
        'p95':    f'{np.percentile(arr, 95):.4f}',
        'Max':    f'{arr.max():.4f}',
        'Skew':   f'{stats.skew(arr):.3f}',
        'Kurt':   f'{stats.kurtosis(arr):.3f}',
    }

stats_df = pd.DataFrame([
    describe(gt,     'GT'),
    describe(pred_a, LABEL_A),
    describe(pred_b, LABEL_B),
    describe(diff,   'Diff (A − B)'),
]).set_index('Model')

display(stats_df.T)

## 7. Prediction distributions

In [ ]:
bins = np.linspace(0, 1, 51)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hist(gt,     bins=bins, alpha=0.5, label='GT',     color='gray',        density=True)
ax.hist(pred_a, bins=bins, alpha=0.6, label=LABEL_A,  color='steelblue',   density=True)
ax.hist(pred_b, bins=bins, alpha=0.6, label=LABEL_B,  color='tomato',      density=True)
ax.set_xlabel('FaceOcclusion'); ax.set_ylabel('Density')
ax.set_title('Prediction distributions'); ax.legend(fontsize=8)

ax = axes[1]
for arr, label, color in [(gt, 'GT', 'gray'), (pred_a, LABEL_A, 'steelblue'), (pred_b, LABEL_B, 'tomato')]:
    s = np.sort(arr); cdf = np.arange(1, len(s)+1) / len(s)
    ax.plot(s, cdf, label=label, color=color)
ax.set_xlabel('FaceOcclusion'); ax.set_ylabel('CDF')
ax.set_title('Empirical CDF'); ax.legend(fontsize=8)

ax = axes[2]
ax.hist(diff, bins=51, color='mediumpurple', alpha=0.8, density=True)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.axvline(diff.mean(), color='red', linewidth=1.5, linestyle='-', label=f'mean={diff.mean():.4f}')
ax.set_xlabel('A − B'); ax.set_ylabel('Density')
ax.set_title('Prediction difference (A − B)'); ax.legend()

plt.tight_layout(); plt.show()

## 8. GT vs prediction scatter (each model)

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(gt), min(5000, len(gt)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, label, color in [
    (axes[0], pred_a, LABEL_A, 'steelblue'),
    (axes[1], pred_b, LABEL_B, 'tomato'),
]:
    r, _ = stats.pearsonr(gt[idx], preds[idx])
    mae  = np.abs(gt[idx] - preds[idx]).mean()
    ax.scatter(gt[idx], preds[idx], alpha=0.2, s=4, color=color)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('GT'); ax.set_ylabel('Pred')
    ax.set_title(f'{label}\nr={r:.3f}, MAE={mae:.4f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout(); plt.show()

## 9. Scatter: A vs B predictions per sample

In [ ]:
r, pval = stats.pearsonr(pred_a, pred_b)
rho, _  = stats.spearmanr(pred_a, pred_b)
mae     = np.abs(diff).mean()
rmse    = np.sqrt((diff ** 2).mean())

print(f'Pearson r        = {r:.4f}  (p={pval:.2e})')
print(f'Spearman ρ       = {rho:.4f}')
print(f'MAE(A, B)        = {mae:.4f}')
print(f'RMSE(A, B)       = {rmse:.4f}')
print(f'A predicts more  = {(pred_a > pred_b).mean()*100:.1f}% of samples')
print(f'B predicts more  = {(pred_b > pred_a).mean()*100:.1f}% of samples')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pred_b[idx], pred_a[idx], alpha=0.2, s=4, color='steelblue')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel(LABEL_B); ax.set_ylabel(LABEL_A)
ax.set_title(f'Per-sample predictions\nr={r:.3f}, ρ={rho:.3f}, MAE={mae:.4f}')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 10. Error by gender and occlusion bucket

In [ ]:
n_buckets = 10
edges = np.linspace(0, 1, n_buckets + 1)
labels_bucket = [f'[{edges[i]:.1f},{edges[i+1]:.1f})' for i in range(n_buckets)]
bucket_idx = np.clip(np.digitize(gt, edges[1:-1]), 0, n_buckets - 1)

rows = []
for i in range(n_buckets):
    mask = bucket_idx == i
    if mask.sum() == 0:
        continue
    for g_val, g_name in [(0.0, 'F'), (1.0, 'M'), (None, 'All')]:
        m = mask if g_val is None else mask & (gender == g_val)
        if m.sum() == 0:
            continue
        se_a = (pred_a[m] - gt[m]) ** 2
        se_b = (pred_b[m] - gt[m]) ** 2
        rows.append({
            'Bucket': labels_bucket[i],
            'Gender': g_name,
            'N': m.sum(),
            'RMSE_A': f'{np.sqrt(se_a.mean()):.4f}',
            'RMSE_B': f'{np.sqrt(se_b.mean()):.4f}',
            'ΔRMSE(A−B)': f'{np.sqrt(se_a.mean()) - np.sqrt(se_b.mean()):+.4f}',
        })

display(pd.DataFrame(rows).set_index(['Bucket', 'Gender']))

## 11. Most divergent samples

In [ ]:
comparison = pd.DataFrame({
    'filename':   df['filename'],
    'gt':         gt,
    LABEL_A:      pred_a,
    LABEL_B:      pred_b,
    'err_A':      np.abs(pred_a - gt),
    'err_B':      np.abs(pred_b - gt),
    'diff (A-B)': diff,
    'abs_diff':   np.abs(diff),
    'gender':     gender,
})

print(f'=== 15 samples where A > B most (A predicts higher) ===')
display(comparison.nlargest(15, 'diff (A-B)')[['filename','gt',LABEL_A,LABEL_B,'diff (A-B)','gender']].reset_index(drop=True))

print(f'\n=== 15 samples where B > A most (B predicts higher) ===')
display(comparison.nsmallest(15, 'diff (A-B)')[['filename','gt',LABEL_A,LABEL_B,'diff (A-B)','gender']].reset_index(drop=True))

print(f'\n=== 15 samples where A has largest absolute error ===')
display(comparison.nlargest(15, 'err_A')[['filename','gt',LABEL_A,LABEL_B,'err_A','err_B','gender']].reset_index(drop=True))

print(f'\n=== 15 samples where B has largest absolute error ===')
display(comparison.nlargest(15, 'err_B')[['filename','gt',LABEL_A,LABEL_B,'err_A','err_B','gender']].reset_index(drop=True))

## 12. Ensemble: average of A and B

In [ ]:
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    pred_mix = alpha * pred_a + (1 - alpha) * pred_b
    s = metric_fn(
        torch.tensor(pred_mix, dtype=torch.float32),
        gt_t, g_t
    ).item()
    label = f'{alpha:.2f}×A + {1-alpha:.2f}×B'
    marker = '  ← best' if s < min(score_a, score_b) else ''
    print(f'{label:<30}  score = {s:.6f}{marker}')

print()
# Fine-grained search
alphas = np.linspace(0, 1, 101)
scores = []
for a in alphas:
    pm = a * pred_a + (1 - a) * pred_b
    scores.append(metric_fn(torch.tensor(pm, dtype=torch.float32), gt_t, g_t).item())

best_a = alphas[np.argmin(scores)]
best_s = min(scores)
print(f'Best ensemble: {best_a:.2f}×A + {1-best_a:.2f}×B  →  score = {best_s:.6f}')
print(f'  vs A alone: {score_a:.6f}  |  B alone: {score_b:.6f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, scores, color='steelblue')
ax.axhline(score_a, color='steelblue', linestyle='--', linewidth=1, label=f'A alone ({score_a:.5f})')
ax.axhline(score_b, color='tomato',    linestyle='--', linewidth=1, label=f'B alone ({score_b:.5f})')
ax.axvline(best_a,  color='green',     linestyle='--', linewidth=1, label=f'best α={best_a:.2f} ({best_s:.5f})')
ax.set_xlabel('α  (weight on A)'); ax.set_ylabel('Competition score')
ax.set_title('Ensemble sweep: α×A + (1−α)×B')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 13. Save best ensemble test predictions

If the ensemble beats both individual models, generate a blended test submission.
Requires that both models have test predictions available.

In [ ]:
# ── Load test predictions ─────────────────────────────────────────────────────
# AMO test predictions (already in csv_to_submit or amo folder)
test_amo_path = Path('test_amo.csv')   # adjust if needed

if not test_amo_path.exists():
    print(f'{test_amo_path} not found — skipping ensemble test CSV generation')
else:
    # Run inference on test set for model B
    test_ds_b = ImageDataset('test', cfg_b, transform_b, augment=False)
    test_loader_b = DataLoader(
        test_ds_b, batch_size=128, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda')
    )

    preds_test_b, filenames_test_b = [], []
    with torch.no_grad():
        offset = 0
        for img, y, gender_t, iw_t, pi_t, gw_t in test_loader_b:
            pred = model_b(img.to(DEVICE)).squeeze(-1).cpu()
            n = len(pred)
            preds_test_b.append(pred)
            filenames_test_b.extend(test_ds_b.filenames[offset : offset + n])
            offset += n

    df_test_b = pd.DataFrame({
        'filename':      filenames_test_b,
        'FaceOcclusion': torch.cat(preds_test_b).numpy(),
        'gender':        'x',
    })

    df_test_a = pd.read_csv(test_amo_path)
    df_test_merged = df_test_a[['filename', 'FaceOcclusion', 'gender']].rename(
        columns={'FaceOcclusion': 'pred_a'}
    ).merge(
        df_test_b[['filename', 'FaceOcclusion']].rename(columns={'FaceOcclusion': 'pred_b'}),
        on='filename'
    )

    df_test_merged['FaceOcclusion'] = best_a * df_test_merged['pred_a'] + (1 - best_a) * df_test_merged['pred_b']
    out = df_test_merged[['filename', 'FaceOcclusion', 'gender']]
    out_path = Path(f'ensemble_a{best_a:.2f}_b{1-best_a:.2f}.csv')
    out.to_csv(out_path, index=False)
    print(f'Saved ensemble test CSV → {out_path}  ({len(out)} rows)')
    print(f'Predictions range: [{out["FaceOcclusion"].min():.4f}, {out["FaceOcclusion"].max():.4f}]')